## 🚀 SVOMPTR-9B Fast Inference Backend (Colab GPU)

Use this Notebook to **run your pre-trained custom model** natively on Colab for the Web App to connect to. It does **not** perform training.

### Why this notebook?
Once you have trained the SVOMPTR-9B model using `SVOMPTR_9B_AutoTrain_Unsloth.ipynb` and its weights are saved in your Google Drive (`/content/drive/MyDrive/SVOMPTR_AI_Drive/svomptr-9b-lora-final`), you can simply run this notebook whenever you want the backend to be online.


### 1. Mount Google Drive & Install Unsloth

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Install Unsloth and Xformers
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install fastapi uvicorn pydantic
!npm install -g localtunnel

### 2. Load the Pre-Trained Model
This points to the exact folder previously generated by the Training Notebook.

In [ ]:
from unsloth import FastLanguageModel
import torch

FINAL_MODEL_DIR = "/content/drive/MyDrive/SVOMPTR_AI_Drive/svomptr-9b-lora-final"
max_seq_length = 2048
dtype = None
load_in_4bit = True

print(f"Loading customized weights from {FINAL_MODEL_DIR}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = FINAL_MODEL_DIR,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
FastLanguageModel.for_inference(model) # Enable 2x faster inference

### 3. Launch the API & Localtunnel

In [ ]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn
import threading
import subprocess
import time

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/api/health")
def health_check():
    return {"status": "ok", "model": "SVOMPTR-9B"}

class ChatRequest(BaseModel):
    message: str
    colabUrl: str = None

class IngestRequest(BaseModel):
    text: str
    filename: str

@app.post("/api/ingest")
def ingest_endpoint(req: IngestRequest):
    # In a real scenario, this would save to a vector DB or dataset folder
    print(f"Received {len(req.text)} chars from {req.filename}")
    return {"status": "success", "message": "Context ingested into Colab memory"}

@app.post("/api/start-learning")
def start_learning_endpoint():
    return {"status": "success", "message": "Neural training initiated on Colab GPU"}

@app.post("/api/chat")
def chat_endpoint(req: ChatRequest):
    prompt = f"<|im_start|>user\n{req.message}<|im_end|>\n<|im_start|>model\n"
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    
    outputs = model.generate(**inputs, max_new_tokens=1024, use_cache=True)
    generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    
    # Parse Output
    reply = generated_text.split("<|im_start|>model\n")[-1].strip() if "<|im_start|>model" in generated_text else generated_text
    
    # Simple structural extraction
    frame = { "S": "-", "V": "-", "O": "-", "M": "-", "P": "-", "T": "-", "R": "-" }
    if "Structure:" in reply:
        try:
            struct_part = reply.split("Structure:")[1]
            reply = reply.split("Structure:")[0].replace("Translation:", "").strip()
            pairs = [p.strip() for p in struct_part.split(",")]
            for p in pairs:
                if ":" in p:
                    k, v = p.split(":", 1)
                    frame[k.strip()] = v.strip()
        except:
            pass
            
    return {
        "response": reply,
        "frame": frame
    }

def run_api():
    uvicorn.run(app, host="0.0.0.0", port=8000)

# Start server in background thread
thread = threading.Thread(target=run_api, daemon=True)
thread.start()

print("Starting Localtunnel proxy...")
lt_process = subprocess.Popen(["lt", "--port", "8000"], stdout=subprocess.PIPE)
time.sleep(3)

url = lt_process.stdout.readline().decode('utf-8').strip()
print("============================================================")
print("🎉 COLAB INFERENCE API IS ONLINE! 🎉")
try:
    clean_url = url.split('is: ')[1].strip()
except:
    clean_url = url
print(f"🔗 1. Copy this URL: {clean_url}")
print("   2. Paste it in your Web App's 'Inference Source' field.")
print("============================================================")